In [ ]:
import pandas as pd
import numpy as np
import os

# Step1 : Intersect HALO_export.csv with allPathassignments07112024.csv where name matches Sherlock_PID


In [ ]:
#get working directory
os.getcwd()
# define working directory
os.chdir ('C:/Users/mutrejak/Documents/VScode/Sherlock/For Github')

In [ ]:

# Read CSV files
Halo_Output = pd.read_csv('C:/Users/mutrejak/Documents/VScode/Sherlock/HALO_export.csv') # add path to your file here
Path_df = pd.read_csv('C:/Users/mutrejak/Documents/VScode/Sherlock/allPathassignments07112024.csv') # add path to your file here

# Perform inner join on 'name' from Halo_Output and 'Sherlock_PID' from Path_df
intersected_df = pd.merge(
    Halo_Output,       # <- this matches HALO_export.csv
    Path_df,           # <- this matches allPathassignments07112024.csv
    left_on='name', 
    right_on='Sherlock_PID', 
    how='inner'
)

# ✅ Display the merged DataFrame
intersected_df.shape


# Intersect with the file that has fixative information 

In [ ]:
fixative_df= pd.read_csv("C:/Users/mutrejak/Documents/VScode/Sherlock/fixative.csv") # add path to your file here 

#g_tissueFixative is a HALO image field, getFixative is a calculated field which combines g_tissueFixative into one row per case
#Fixassign is a calculated field which simplifies getFixative into 3 categories
#if all images were Formalin the Fixassign is Fromalin 
#if they were Formalin and frozen then mixed and if they were frozen only then frozen

fixative_df = fixative_df.drop(columns=['g_tissueFixative']) 
fixative_df = fixative_df.drop_duplicates() #drop duplicates since it was fixative data was image level
df = pd.merge(
    intersected_df,      
    fixative_df,           
    left_on='name', 
    right_on='g_subjectExtID', 
    how='inner'
)
df.shape


In [ ]:
# Remove rows where P1 is NaN, empty, or only whitespace (just a check)
df = df[df['P1'].notna() & (df['P1'].astype(str).str.strip() != "")]
df.shape

# Now are Dataframe is ready

## Create Header comparison column

In [ ]:
def header_comparison(row):
    P3 = row['P3']
    P1 = row['DCEG_Shlck_header_Diag_P1']
    P2 = row['DCEG_Shlck_header_Diag_P2']
    P3_val = row['DCEG_Shlck_header_Diag_P3']

    # CASE 1: P3 == 'None'
    if P3 == 'None':
        if P1 == P2:
            return 'all_agree(2)'
        else:
            return 'no_agreement(additional review)'

    # CASE 2: P3 != 'None'
    elif P3 != 'None':
        if P1 == P2:
            if P1 == P3_val:
                return 'all_agree(3)'
            else:
                return 'all_agree(2)'
        elif P1 != P2:
            if P1 == P3_val:
                return 'all_agree(2)'
            elif P2 == P3_val:
                return 'all_agree(2)'
            else:
                return 'no_agreement(additional review)'

    # Default case
    return 'no_agreement(additional review)'

# Apply the function row-wise to create new column
df['header_comparison_test'] = df.apply(header_comparison, axis=1)


In [ ]:
df['header_comparison_test'].value_counts()

In [ ]:
df.isna().sum().reset_index()

In [ ]:
def s(x):
    """Safely convert to trimmed string (handles NaN, None, etc.)."""
    return "" if pd.isna(x) else str(x).strip()

def check1_p1_headerC(row):
    header = s(row['DCEG_Shlck_header_Diag_P1'])
    single = s(row['DCEG_Shrlck_HistoTypeSingle_P1'])
    mixed1 = s(row['DCEG_Shrlck_HistoTypeMixed1_P1'])
    mixed2 = s(row['DCEG_Shrlck_HistoTypeMixed2_P1'])
    noeval = s(row['DCEG_Shrlck_HistoTypeNoEval_P1'])

    # Option A
    if header == 'Option A: Tumor with single histology (proceed to option A below)':
        if (single == mixed1) or (mixed1 == '') or (single == mixed2):
            return single
        else:
            return 'QC FAIL'

    # Option B
    elif header == 'Option B: Tumor with combined/mixed histology (proceed to option B below)':
        if mixed1 == mixed2:
            return 'QC FAIL'
        elif (single == mixed1) or (single == mixed2):
            return mixed1 + mixed2
        else:
            return mixed1 + mixed2

    # Option C (using header)
    elif header == 'Option C: Not evaluable (proceed to option C below)':
        if (single == 'Adenocarcinoma, non-mucinous') or (mixed1 == '') or (mixed2 == ''):
            return 'QC FAIL'
        else:
            return 'QC FAIL'

    else:
        return 'QC FAIL'

df['Check1_P1'] = df.apply(check1_p1_headerC, axis=1)
#print(df['Check1_P1'].value_counts(dropna=False))


In [ ]:
#def s(x):
#    """Safely convert to trimmed string (handles NaN, None, etc.)."""
#    return "" if pd.isna(x) else str(x).strip()

def check1_p2_headerC(row):
    header = s(row['DCEG_Shlck_header_Diag_P2'])
    single = s(row['DCEG_Shrlck_HistoTypeSingle_P2'])
    mixed1 = s(row['DCEG_Shrlck_HistoTypeMixed1_P2'])
    mixed2 = s(row['DCEG_Shrlck_HistoTypeMixed2_P2'])
    noeval = s(row['DCEG_Shrlck_HistoTypeNoEval_P2'])

    # Option A
    if header == 'Option A: Tumor with single histology (proceed to option A below)':
        if (single == mixed1) or (mixed1 == '') or (single == mixed2):
            return single
        else:
            return 'QC FAIL'

    # Option B
    elif header == 'Option B: Tumor with combined/mixed histology (proceed to option B below)':
        if mixed1 == mixed2:
            return 'QC FAIL'
        elif (single == mixed1) or (single == mixed2):
            return mixed1 + mixed2
        else:
            return mixed1 + mixed2

    # Option C (using header)
    elif header == 'Option C: Not evaluable (proceed to option C below)':
        if (single == 'Adenocarcinoma, non-mucinous') or (mixed1 == '') or (mixed2 == ''):
            return 'QC FAIL'
        else:
            return 'QC FAIL'

    else:
        return 'QC FAIL'

df['Check1_P2'] = df.apply(check1_p2_headerC, axis=1)
#print(df['Check1_P2'].value_counts(dropna=False))


In [ ]:
#def s(x):
#    """Safely convert to trimmed string."""
#    return "" if pd.isna(x) else str(x).strip()

def check1_p3_final(row):
    # Clean up values
    P3      = s(row.get('P3'))
    header  = s(row.get('DCEG_Shlck_header_Diag_P3'))
    single  = s(row.get('DCEG_Shrlck_HistoTypeSingle_P3'))
    mixed1  = s(row.get('DCEG_Shrlck_HistoTypeMixed1_P3'))
    mixed2  = s(row.get('DCEG_Shrlck_HistoTypeMixed2_P3'))
    noeval  = s(row.get('DCEG_Shrlck_HistoTypeNoEval_P3'))

    # When P3 is missing, blank, or literal 'None'
    if P3 in ["", "None", "none", "NONE"]:
        return "P3 not assigned"

    # When P3 has some other value
    # Option A
    if header == 'Option A: Tumor with single histology (proceed to option A below)':
        if (single == mixed1) or (mixed1 == '') or (single == mixed2):
            return single
        else:
            return 'QC FAIL'

    # Option B
    elif header == 'Option B: Tumor with combined/mixed histology (proceed to option B below)':
        if (single == '') or (single == mixed1) or (single == mixed2):
            return mixed1 + mixed2
        else:
            return 'QC FAIL'

    # Option C (based on single column)
    elif header == 'Option C: Not evaluable (proceed to option C below)':
        if (single == '') or (mixed1 == '') or (mixed2 == ''):
            return noeval
        else:
            return 'QC FAIL'

    # Fallback
    else:
        return 'QC FAIL'


#Apply to DataFrame
df['Check1_P3'] = df.apply(check1_p3_final, axis=1)

# Summary
#print(df['Check1_P3'].value_counts(dropna=False))

### Accumalate all types of carcinoid into one category

In [ ]:
# Replace Carcinoid variants with "Carcinoid"
cols_to_update = [
    'DCEG_Shrlck_HistoTypeSingle_P1', 'DCEG_Shrlck_HistoTypeSingle_P2', 'DCEG_Shrlck_HistoTypeSingle_P3',
    'DCEG_Shrlck_HistoTypeMixed1_P1', 'DCEG_Shrlck_HistoTypeMixed1_P2', 'DCEG_Shrlck_HistoTypeMixed1_P3',
    'DCEG_Shrlck_HistoTypeMixed2_P1', 'DCEG_Shrlck_HistoTypeMixed2_P2', 'DCEG_Shrlck_HistoTypeMixed2_P3',
    'Check1_P1', 'Check1_P2', 'Check1_P3'
]

# Define the mapping (Step 9–11)
replace_map = {
    'Carcinoid, typical': 'Carcinoid',
    'Carcinoid, atypical': 'Carcinoid',
    'Carcinoid, NOS': 'Carcinoid'
}

# Apply replacements across all specified columns
df[cols_to_update] = df[cols_to_update].replace(replace_map)

#  Verify results
for c in cols_to_update:
    print(f"{c}: {(df[c].isin(replace_map.keys())).sum()} remaining variant values")


### Create filter for Adenocarcinoma

In [ ]:
def s(x):
    """Safely convert to trimmed string."""
    return "" if pd.isna(x) else str(x).strip()

def adeno_filter(row):
    P3 = s(row['P3'])
    p1 = s(row['Check1_P1'])
    p2 = s(row['Check1_P2'])
    p3 = s(row['Check1_P3'])

    # WHEN P3 = 'None' → only P1 and P2
    if P3 == 'None':
        return f"{p1} ; {p2}"

    # WHEN P3 is not 'None' → include P3
    elif P3 != '':
        return f"{p1} ; {p2} ; {p3}"

    # Optional fallback (if P3 is blank or missing)
    else:
        return f"{p1} ; {p2}"

df['Adeno_filter'] = df.apply(adeno_filter, axis=1)

# ✅ Check result
df[['P3', 'Check1_P1', 'Check1_P2', 'Check1_P3', 'Adeno_filter']].head()


### Primary Histology Comparison by all pathologists

In [ ]:

# already written this function above but just keep on repeating it out of habbit
def s(x):
    """Safely convert to trimmed string (handles NaN/None)."""
    return "" if pd.isna(x) else str(x).strip()

def primary_histology_comparison(row):
    P3  = s(row.get('P3'))
    c1  = s(row.get('Check1_P1'))
    c2  = s(row.get('Check1_P2'))
    c3  = s(row.get('Check1_P3'))
    af  = s(row.get('Adeno_filter'))

    # Treat 'None', '', NaN, or spaces as "like None"
    if P3 in ["", "None", "none", "NONE"]:
        # CASE: P3 == 'None'
        if c1 == c2:
            return 'all_agree(2)'
        elif c1 != c2:
            # Check for 'Adenocarcinoma, non-mucinous' appearing twice
            if af.count('Adenocarcinoma, non-mucinous') >= 2:
                return 'all_agree(2)'
            else:
                return 'no_agreement(additional review)'
        else:
            return 'no_agreement(additional review)'

    # ✅ CASE: P3 is not 'None'
    else:
        if c1 != c2:
            if (c1 == c3) or (c2 == c3):
                return 'all_agree(2)'
            else:
                return 'no_agreement(additional review)'
        elif c1 == c2:
            if c1 == c3:
                return 'all_agree(3)'
            else:
                return 'all_agree(2)'
        else:
            return 'no_agreement(additional review)'

# ✅ Create new column
df['Primary_histology_comparison'] = df.apply(primary_histology_comparison, axis=1)

# ✅ Summary
print(df['Primary_histology_comparison'].value_counts(dropna=False))

#### Calculate the total percentage of each subtype should come out to be 100.

In [ ]:

# Ensure all nulls are replaced by 0 before summing (in case you haven’t already)
cols_p1 = [
    'DCEG_Shrlck_subt_lepdic_pct_P1',
    'DCEG_Shrlck_subt_acinar_pct_P1',
    'DCEG_Shrlck_subt_pap_pct_P1',
    'DCEG_Shrlck_subt_mpap_pct_P1',
    'DCEG_Shrlck_subt_solid_pct_P1',
    'DCEG_Shrlck_subt_crib_pct_P1'
]
cols_p2 = [
    'DCEG_Shrlck_subt_lepdic_pct_P2',
    'DCEG_Shrlck_subt_acinar_pct_P2',
    'DCEG_Shrlck_subt_crib_pct_P2',
    'DCEG_Shrlck_subt_pap_pct_P2',
    'DCEG_Shrlck_subt_mpap_pct_P2',
    'DCEG_Shrlck_subt_solid_pct_P2'
]
cols_p3 = [
    'DCEG_Shrlck_subt_lepdic_pct_P3',
    'DCEG_Shrlck_subt_acinar_pct_P3',
    'DCEG_Shrlck_subt_crib_pct_P3',
    'DCEG_Shrlck_subt_pap_pct_P3',
    'DCEG_Shrlck_subt_mpap_pct_P3',
    'DCEG_Shrlck_subt_solid_pct_P3'
]

# Convert to numeric (integer) safely and replace NaN with 0
for col in cols_p1 + cols_p2 + cols_p3:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Step 17: total of subtype path 1
df['getTotal_subtype_p1'] = df[cols_p1].sum(axis=1)

# Step 18: total of subtype path 2
df['getTotal_subtype_p2'] = df[cols_p2].sum(axis=1)

# Step 19: total of subtype path 3
df['getTotal_subtype_p3'] = df[cols_p3].sum(axis=1)

# ✅ Preview results
print(df[['getTotal_subtype_p1', 'getTotal_subtype_p2', 'getTotal_subtype_p3']].head())


#### code in the cell below is just a check to get number of rows that are not equal to 100 it doesnot change the outcome and can be ignored

In [ ]:
# Count how many totals are NOT equal to 100 for each path
not_100_p1 = (df['getTotal_subtype_p1'] != 100).sum()
not_100_p2 = (df['getTotal_subtype_p2'] != 100).sum()
not_100_p3 = (df['getTotal_subtype_p3'] != 100).sum()

# ✅ Print summary
print("Rows where totals ≠ 100:")
print(f"  P1: {not_100_p1}")
print(f"  P2: {not_100_p2}")
print(f"  P3: {not_100_p3}")

# Optional — show how many were exactly 100
print("\nRows where totals = 100:")
print(f"  P1: {(df['getTotal_subtype_p1'] == 100).sum()}")
print(f"  P2: {(df['getTotal_subtype_p2'] == 100).sum()}")
print(f"  P3: {(df['getTotal_subtype_p3'] == 100).sum()}")


In [ ]:
# Convert these columns to numeric (coerce bad strings → NaN → 0)
grade_cols = [
    'DCEG_Shrlck_subt_solid_pct_P1', 'DCEG_Shrlck_subt_mpap_pct_P1', 'DCEG_Shrlck_subt_crib_pct_P1',
    'DCEG_Shrlck_subt_solid_pct_P2', 'DCEG_Shrlck_subt_mpap_pct_P2', 'DCEG_Shrlck_subt_crib_pct_P2',
    'DCEG_Shrlck_subt_solid_pct_P3', 'DCEG_Shrlck_subt_mpap_pct_P3', 'DCEG_Shrlck_subt_crib_pct_P3'
]
for c in grade_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0).astype(int)

# Step 19: High_gradeP1
df['High_gradeP1'] = (
    df['DCEG_Shrlck_subt_solid_pct_P1'] +
    df['DCEG_Shrlck_subt_mpap_pct_P1'] +
    df['DCEG_Shrlck_subt_crib_pct_P1']
)

# Step 20: High_gradeP2
df['High_gradeP2'] = (
    df['DCEG_Shrlck_subt_solid_pct_P2'] +
    df['DCEG_Shrlck_subt_mpap_pct_P2'] +
    df['DCEG_Shrlck_subt_crib_pct_P2']
)

# Step 21: High_gradeP3
df['High_gradeP3'] = (
    df['DCEG_Shrlck_subt_solid_pct_P3'] +
    df['DCEG_Shrlck_subt_mpap_pct_P3'] +
    df['DCEG_Shrlck_subt_crib_pct_P3']
)

# Preview the results
print(df[['High_gradeP1', 'High_gradeP2', 'High_gradeP3']].head())

#### Calculate Acinar + Pap this will be used to determine differentiation status

In [ ]:
# Ensure columns are numeric and replace NaN with 0
for col in [
    'DCEG_Shrlck_subt_acinar_pct_P1', 'DCEG_Shrlck_subt_pap_pct_P1',
    'DCEG_Shrlck_subt_acinar_pct_P2', 'DCEG_Shrlck_subt_pap_pct_P2',
    'DCEG_Shrlck_subt_acinar_pct_P3', 'DCEG_Shrlck_subt_pap_pct_P3'
]:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Create new columns
df['Acinar+Pap_P1'] = df['DCEG_Shrlck_subt_acinar_pct_P1'] + df['DCEG_Shrlck_subt_pap_pct_P1']
df['Acinar+pap_P2'] = df['DCEG_Shrlck_subt_acinar_pct_P2'] + df['DCEG_Shrlck_subt_pap_pct_P2']
df['Acinar+pap_P3'] = df['DCEG_Shrlck_subt_acinar_pct_P3'] + df['DCEG_Shrlck_subt_pap_pct_P3']

# Preview results
print(df[['Acinar+Pap_P1', 'Acinar+pap_P2', 'Acinar+pap_P3']].head())

### Calculate Differentiation status of each case by P1, P2 and P3

In [ ]:
def differentiation_p1(row):
    hg  = row['High_gradeP1']
    lep = row['DCEG_Shrlck_subt_lepdic_pct_P1']
    acp = row['Acinar+Pap_P1']

    if hg >= 20:
        return 'poorly'
    elif hg < 20:
        if lep >= acp:
            return 'well'
        elif lep < acp:
            return 'moderate'
    return None  # for any undefined case


def differentiation_p2(row):
    hg  = row['High_gradeP2']
    lep = row['DCEG_Shrlck_subt_lepdic_pct_P2']
    acp = row['Acinar+pap_P2']

    if hg >= 20:
        return 'poorly'
    elif hg < 20:
        if lep >= acp:
            return 'well'
        elif lep < acp:
            return 'moderate'
    return None


def differentiation_p3(row):
    hg  = row['High_gradeP3']
    lep = row['DCEG_Shrlck_subt_lepdic_pct_P3']
    acp = row['Acinar+pap_P3']

    if hg >= 20:
        return 'poorly'
    elif hg < 20:
        if lep >= acp:
            return 'well'
        elif lep < acp:
            return 'moderate'
    return None


# Apply functions to create new columns
df['DifferentiationP1'] = df.apply(differentiation_p1, axis=1)
df['DifferentiationP2'] = df.apply(differentiation_p2, axis=1)
df['DifferentiationP3'] = df.apply(differentiation_p3, axis=1)

# Preview
print(df[['DifferentiationP1', 'DifferentiationP2', 'DifferentiationP3']].head())

# Optional summary counts
print("\nDifferentiation summary:")
print(df[['DifferentiationP1', 'DifferentiationP2', 'DifferentiationP3']].apply(pd.Series.value_counts))


### Differentiation status only calculated for adenocarcinoma cases

In [ ]:
# Step: update DifferentiationP1
df['DifferentiationP1'] = df.apply(
    lambda r: 'Not calculated1'
    if str(r['Check1_P1']).strip() != 'Adenocarcinoma, non-mucinous'
    else r['DifferentiationP1'],
    axis=1
)

# Step: update DifferentiationP2
df['DifferentiationP2'] = df.apply(
    lambda r: 'Not calculated2'
    if str(r['Check1_P2']).strip() != 'Adenocarcinoma, non-mucinous'
    else r['DifferentiationP2'],
    axis=1
)

# Step: update DifferentiationP3
df['DifferentiationP3'] = df.apply(
    lambda r: 'Not calculated3'
    if str(r['Check1_P3']).strip() != 'Adenocarcinoma, non-mucinous'
    else r['DifferentiationP3'],
    axis=1
)

# Preview
print(df[['Check1_P1','DifferentiationP1','Check1_P2','DifferentiationP2','Check1_P3','DifferentiationP3']].head())


#### Compare Differentiation 

In [ ]:
import pandas as pd

def compare_differentiation(row):
    d1 = str(row.get('DifferentiationP1', '')).strip()
    d2 = str(row.get('DifferentiationP2', '')).strip()
    d3 = str(row.get('DifferentiationP3', '')).strip()

    # CASE 1: DifferentiationP1 == DifferentiationP2
    if d1 == d2:
        if d2 != d3:
            return 'all_agree(2)'
        else:
            return 'all_agree(3)'

    # CASE 2: DifferentiationP1 != DifferentiationP2
    elif d1 != d2:
        if d1 == d3:
            return 'all_agree(2)'
        elif d2 == d3:
            return 'all_agree(2)'
        else:
            return 'no_agreement(additional review)'

    # Default (shouldn't normally happen)
    return 'no_agreement(additional review)'


# Apply function to create new column
df['Compare_differentiation'] = df.apply(compare_differentiation, axis=1)

# View summary of results
print(df['Compare_differentiation'].value_counts(dropna=False))

# Preview few rows
print(df[['DifferentiationP1', 'DifferentiationP2', 'DifferentiationP3', 'Compare_differentiation']].head())


In [ ]:
# Create Diss_P1
df['Diss_P1'] = df['DCEG_Shrlck_add_discussion_P1'].apply(
    lambda x: 0 if str(x).strip() == 'No' else 1
)

# Create Diss_P2
df['Diss_P2'] = df['DCEG_Shrlck_add_discussion_P2'].apply(
    lambda x: 0 if str(x).strip() == 'No' else 1
)

# Create Diss_P3
df['Diss_P3'] = df['DCEG_Shrlck_add_discussion_P3'].apply(
    lambda x: 0 if str(x).strip() == 'No' else 1
)

# ✅ Add total column
df['NumberofPath'] = df['Diss_P1'] + df['Diss_P2'] + df['Diss_P3']

# ✅ Preview results
print(df[['DCEG_Shrlck_add_discussion_P1','Diss_P1',
          'DCEG_Shrlck_add_discussion_P2','Diss_P2',
          'DCEG_Shrlck_add_discussion_P3','Diss_P3',
          'NumberofPath']].head())

# ✅ Optional summary
print("\nSummary of NumberofPath:")
print(df['NumberofPath'].value_counts(dropna=False))

## Resolution column


In [ ]:


# ---------- helpers ----------
def s(x):
    """Safe, trimmed string (NaN/None -> '')."""
    return "" if pd.isna(x) else str(x).strip()

def resolution(row):
    # pull + normalize strings
    c1  = s(row.get("Check1_P1"))
    c2  = s(row.get("Check1_P2"))
    c3  = s(row.get("Check1_P3"))
    P3v = s(row.get("P3"))
    phc = s(row.get("Primary_histology_comparison"))
    hct = s(row.get("header_comparison_test"))
    cdf = s(row.get("Compare_differentiation"))
    af  = s(row.get("Adeno_filter"))

    # raw (not cast) totals & review flags
    g1  = row.get("getTotal_subtype_p1")
    g2  = row.get("getTotal_subtype_p2")
    g3  = row.get("getTotal_subtype_p3")
    rc1 = row.get("DCEG_Shrlck_review_complete_P1")
    rc2 = row.get("DCEG_Shrlck_review_complete_P2")
    rc3 = row.get("DCEG_Shrlck_review_complete_P3")

    # discuss flags (checked at the very end per your SQL order)
    d1  = s(row.get("DCEG_Shrlck_add_discussion_P1"))
    d2  = s(row.get("DCEG_Shrlck_add_discussion_P2"))
    d3  = s(row.get("DCEG_Shrlck_add_discussion_P3"))

    # ---- Top-level QC FAIL checks ----
    if c1 == "QC FAIL" or c2 == "QC FAIL" or c3 == "QC FAIL":
        return "QC FAIL"

    # Determine P3 bucket: None-like vs other
    p3_none_like = (P3v == "") or (P3v.lower() == "none")

    # ====================== Branch: P3 is None-like ======================
    if p3_none_like:
        if pd.isna(rc1):
            if pd.isna(rc2):
                return "Pending review(P1 and P2)"
            else:
                return "Pending review(P1)"
        if pd.isna(rc2):
            if pd.isna(rc1):
                return "Pending review(P1 and P2)"
            else:
                return "Pending review(P2)"

        if phc == "no_agreement(additional review)":
            return "Fix Histology"

        if af != "Adenocarcinoma, non-mucinous ; Adenocarcinoma, non-mucinous":
            return "Not Adenocarcinoma"

        if g1 != 100:
            return "Fix subtype"
        if g2 != 100:
            return "Fix subtype"

        if cdf.startswith("no_agreement"):
            if phc.startswith("all_agree"):
                if af == "Adenocarcinoma, non-mucinous ; Adenocarcinoma, non-mucinous":
                    return "Assign P3"

        if cdf == "all_agree(2)":
            if hct == "all_agree(2)":
                if af == "Adenocarcinoma, non-mucinous ; Adenocarcinoma, non-mucinous":
                    return "P1/P2 Diff."
                else:
                    return "Not Adenocarcinoma"
            else:
                return "P1/P2 discordant on single Histology but primary histology matches"

        if phc == "all_agree(2)":
            if af == "Adenocarcinoma, non-mucinous ; Adenocarcinoma, non-mucinous":
                return "Complete"
            else:
                return "Not Adenocarcinoma"

        return "no category1"

    # =================== Branch: P3 is NOT None-like ====================
    else:
        if pd.isna(rc3):
            return "Pending review(P3)"

        if phc == "no_agreement(additional review)":
            return "Fix Histology"

        if cdf.startswith("all_agree"):
            if hct.startswith("all_agree"):
                if af.count("Adenocarcinoma, non-mucinous") >= 2:
                    return "P1/P2/P3(atleast2) Diff."
                else:
                    return "Not Adenocarcinoma"
            else:
                return "P1/P2/P3 discordant on single Histology"

        if cdf == "no_agreement(additional review)":
            if af.count("Adenocarcinoma, non-mucinous") >= 2:
                return "Primary Histology match (atleast2); diff. disagreement"
            if hct.startswith("all_agree"):
                if phc.startswith("all_agree"):
                    if af.count("Adenocarcinoma, non-mucinous") >= 2:
                        return "Primary Histology match (atleast2); diff. disagreement"
                    else:
                        return "Not Adenocarcinoma"
            else:
                return "P1/P2/P3 discordant on single histology"

        if g1 != 100:
            return "Fix subtype"
        if g2 != 100:
            return "Fix subtype"
        if g3 != 100:
            return "Fix subtype"

        if af != "Adenocarcinoma, non-mucinous ; Adenocarcinoma, non-mucinous ; Adenocarcinoma, non-mucinous":
            return "Not Adenocarcinoma"

        return "no category1"

    # (Per your SQL these are after the main branches; unreachable given returns above)
    #if d1 == "Yes" or d2 == "Yes" or d3 == "Yes":
    #return "no category"

# ---------- build Resolution from scratch ----------
df['Resolution'] = df.apply(resolution, axis=1)

# ---------- post-processing overwrite rule ----------
# Treat P3 None-like (None/blank/empty/NaN) vs other
P3_str = df['P3'].astype(str).str.strip()
p3_none_like = P3_str.eq('') | P3_str.str.casefold().eq('none') | df['P3'].isna()
c1 = df['Check1_P1'].astype(str).str.strip()
c2 = df['Check1_P2'].astype(str).str.strip()
c3 = df['Check1_P3'].astype(str).str.strip()

m_base = df['Resolution'].astype(str).str.strip().eq('Not Adenocarcinoma')

# A) P3 None-like & P1 != P2  -> Fix Histology
m_fix_A = m_base & p3_none_like & (c1 != c2)

# B) P3 other & P1 != P2 & P3 matches neither P1 nor P2 -> Fix Histology
m_fix_B = m_base & (~p3_none_like) & (c1 != c2) & (c1 != c3) & (c2 != c3)

df.loc[m_fix_A | m_fix_B, 'Resolution'] = 'Fix Histology'
print(df['Resolution'].value_counts(dropna=False))

In [ ]:

# def s(x):
#    """Safely convert to trimmed string (handles NaN/None)."""
#    return "" if pd.isna(x) else str(x).strip()

def get_status(row):
    res = s(row.get("Resolution"))
    phc = s(row.get("Primary_histology_comparison"))

    # --- Direct Resolved cases ---
    if res == "Complete":
        return "Resolved"
    elif res == "P1/P2 Diff.":
        return "Resolved"
    elif res == "P1/P2/P3(atleast2) Diff.":
        return "Resolved"

    # --- Direct Unresolved cases ---
    elif res in [
        "Assign P3", "Fix Histology", "Fix subtype", "QC FAIL",
        "Primary Histology match (atleast2); diff. disagreement",
        "P1/P2 discordant on single Histology but primary histology matches",
        "Pending review(P1)", "Pending review(P2)",
        "Pending review(P3)", "Pending review(P1 and P2)"
    ]:
        return "Unresolved"

    # --- Conditional logic for Not Adenocarcinoma ---
    elif res == "Not Adenocarcinoma":
        if phc.startswith("all_agree"):
            return "Resolved"
        else:
            return "Unresolved"

    # --- Default ---
    else:
        return "no category"

# ✅ Apply to DataFrame
df['Status'] = df.apply(get_status, axis=1)

df['DCEG_Shrlck_study_dir_resolve'] = (
    df['DCEG_Shrlck_study_dir_resolve']
    .fillna('')                       # replace NaN with empty string
    .astype(str)
    .str.strip()                      # remove spaces around text
)

# 2️⃣ Update Status
df['Status'] = df.apply(
    lambda r: (
        r['Status']                    # if the field is blank, keep existing
        if r['DCEG_Shrlck_study_dir_resolve'] == ''
        else 'Resolved with comment'   # if non-empty, override
    ),
    axis=1
)

# ✅ Optional: Verify
print(df['Status'].value_counts(dropna=False))

### calculate differentiation result

In [ ]:
# 1️⃣ Replace empty cells in 'DCEG_Shrlck_study_dir_resolve' and normalize
df['DCEG_Shrlck_study_dir_resolve'] = (
    df['DCEG_Shrlck_study_dir_resolve']
    .fillna('')
    .astype(str)
    .str.strip()
)

# 2️⃣ Update Status — only when there's a non-empty comment
df.loc[
    df['DCEG_Shrlck_study_dir_resolve'].ne('') &
    df['Status'].astype(str).str.strip().eq('no category'),
    'Status'
] = 'Resolved with comment'

# 3️⃣ Create combine_differentiation column
df['combine_differentiation'] = (
    df['DifferentiationP1'].astype(str).str.strip() + ',' +
    df['DifferentiationP2'].astype(str).str.strip() + ',' +
    df['DifferentiationP3'].astype(str).str.strip()
)

# 4️⃣ Create Differentiation_result1 column based on combined text
def get_diff_result(x):
    x = str(x).lower()
    if x.count('well') >= 2:
        return 'well'
    elif x.count('moderate') >= 2:
        return 'moderate'
    elif x.count('poorly') >= 2:
        return 'poorly'
    elif x.count('not calculated') >= 2:
        return 'Not Calculated'
    else:
        return 'no_agreement'

df['Differentiation_result1'] = df['combine_differentiation'].apply(get_diff_result)

# ✅ Optional: Quick summary check

print(df['Differentiation_result1'].value_counts(dropna=False))

# Lead Pathologist overrides the differentiation result

In [ ]:
# Define all hard-coded name → Resolution mappings
# These ovverrides are optional and below are the ones that we had done in our study please adjust as needed
diff_result_overrides = {
    'NSLC-IARC-0043': 'poorly',
    'NSLC-LAVA-0100': 'well',
    'NSLC-MAYO-0075': 'Not Calculated',
    'NSLC-TWAN-0008': 'poorly',
    'NSLC-TWAN-0086': 'Not Calculated',
    'NSLC-TWAN-0154': 'poorly',
    'NSLC-YALE-0003': 'poorly',
    'NSLC-YALE-0021': 'poorly',
    'NSLC-YALE-0032': 'poorly'
}

# 2️⃣ Add new column Differentiation_result
df['Differentiation_result'] = (
    df['name'].map(diff_result_overrides)   # map special names
    .fillna(df['Differentiation_result1'])  # otherwise use Differentiation_result1
)

# ✅ Optional check
print(df[['name', 'Differentiation_result']].head(15))

# Lead Pathologist overrides the resolution

In [ ]:
# Define all hard-coded name → Resolution mappings
# These ovverrides are optional and below are the ones that we had done in our study please adjust as needed
resolution_map = {
    'NSLC-FVDL-0006': 'Complete/Hist Hard Coded',
    'NSLC-IARC-0001': 'Complete/Hist Hard Coded',
    'NSLC-IARC-0033': 'Complete/Hist Hard Coded',
    'NSLC-IARC-0043': 'Complete/Diff&Hist Coded',
    'NSLC-IARC-0045': 'Complete/Hist Hard Coded',
    'NSLC-IARC-0073': 'Complete/Hist Hard Coded',
    'NSLC-IARC-0092': 'Complete/Hist Hard Coded',
    'NSLC-IARC-0094': 'Complete/Hist Hard Coded',
    'NSLC-IARC-0116': 'Complete/Diff&Hist Coded',
    'NSLC-IARC-0151': 'Complete/Hist Hard Coded',
    'NSLC-IARC-0166': 'Complete/Hist Hard Coded',
    'NSLC-ITLU-0021': 'Complete/Hist Hard Coded',
    'NSLC-ITLU-0024': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0009': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0020': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0027': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0033': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0035': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0044': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0056': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0062': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0092': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0093': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0100': 'Complete/Diff&Hist Coded',
    'NSLC-LAVA-0115': 'Complete/Diff&Hist Coded',
    'NSLC-MAYO-0075': 'Complete/Diff&Hist Coded',
    'NSLC-MOFF-0003': 'Complete/Diff&Hist Coded',
    'NSLC-NICE-0001': 'Complete/Hist Hard Coded',
    'NSLC-NICE-0022': 'Complete/Hist Hard Coded',
    'NSLC-NICE-0045': 'Complete/Hist Hard Coded',
    'NSLC-NICE-0056': 'Complete/Hist Hard Coded',
    'NSLC-NICE-0101': 'Complete/Hist Hard Coded',
    'NSLC-RONT-0056': 'Complete/Hist Hard Coded',
    'NSLC-TWAN-0008': 'Complete/Diff&Hist Coded',
    'NSLC-TWAN-0047': 'Complete/Hist Hard Coded',
    'NSLC-TWAN-0064': 'Complete/Hist Hard Coded',
    'NSLC-TWAN-0076': 'Complete/Hist Hard Coded',
    'NSLC-TWAN-0086': 'Complete/Diff&Hist Coded',
    'NSLC-TWAN-0115': 'Complete/Hist Hard Coded',
    'NSLC-TWAN-0148': 'Complete/Hist Hard Coded',
    'NSLC-TWAN-0154': 'Complete/Diff&Hist Coded',
    'NSLC-TWAN-0159': 'Complete/Hist Hard Coded',
    'NSLC-TWAN-0205': 'Complete/Diff&Hist Coded',
    'NSLC-YALE-0003': 'Complete/Diff&Hist Coded',
    'NSLC-YALE-0021': 'Complete/Diff&Hist Coded',
    'NSLC-YALE-0032': 'Complete/Diff&Hist Coded',
    'NSLC-BARR-0010': 'Complete/Diff Hard Coded',
    'NSLC-FVDL-0021': 'Complete/Diff Hard Coded',
    'NSLC-HARV-0017': 'Complete/Diff Hard Coded',
    'NSLC-IARC-0080': 'Complete/Diff Hard Coded',
    'NSLC-LAVA-0123': 'Complete/Diff Hard Coded',
    'NSLC-MAYO-0012': 'Complete/Diff Hard Coded',
    'NSLC-MAYO-0050': 'Complete/Diff Hard Coded',
    'NSLC-MAYO-0073': 'Complete/Diff Hard Coded',
    'NSLC-MAYO-0087': 'Complete/Diff Hard Coded',
    'NSLC-MDGM-0026': 'Complete/Diff Hard Coded',
    'NSLC-MDGM-0035': 'Complete/Diff Hard Coded',
    'NSLC-NICE-0023': 'Complete/Diff Hard Coded',
    'NSLC-NICE-0068': 'Complete/Diff Hard Coded',
    'NSLC-NICE-0080': 'Complete/Diff Hard Coded',
    'NSLC-NICE-0081': 'Complete/Diff Hard Coded',
    'NSLC-NICE-0087': 'Complete/Diff Hard Coded',
    'NSLC-NICE-0092': 'Complete/Diff Hard Coded',
    'NSLC-NICE-0102': 'Complete/Diff Hard Coded',
    'NSLC-NICE-0107': 'Complete/Diff Hard Coded',
    'NSLC-RONT-0002': 'Complete/Diff Hard Coded',
    'NSLC-RONT-0093': 'Complete/Diff Hard Coded',
    'NSLC-TWAN-0069': 'Complete/Diff Hard Coded',
    'NSLC-TWAN-0116': 'Complete/Diff Hard Coded',
    'NSLC-VLEN-0019': 'Complete/Diff Hard Coded',
    'NSLC-VLEN-0020': 'Complete/Diff Hard Coded',
    'NSLC-YALE-0004': 'Complete/Diff Hard Coded',
    'NSLC-YALE-0010': 'Complete/Diff Hard Coded',
    'NSLC-YALE-0013': 'Complete/Diff Hard Coded',
    'NSLC-YALE-0016': 'Complete/Diff Hard Coded',
    'NSLC-YALE-0023': 'Complete/Diff Hard Coded',
    'NSLC-YALE-0037': 'Complete/Diff Hard Coded',
    'NSLC-AUST-0004': 'Complete/Hist Hard Coded',
    'NSLC-AUST-0006': 'Complete/Diff Hard Coded',
    'NSLC-AUST-0008': 'Complete/Hist Hard Coded',
    'NSLC-AUST-0014': 'Complete/Diff Hard Coded',
    'NSLC-AUST-0021': 'Complete/Diff Hard Coded',
    'NSLC-AUST-0023': 'Complete/Diff Hard Coded',
    'NSLC-AUST-0036': 'Complete/Hist Hard Coded',
    'NSLC-BARR-0009': 'Complete/Hist Hard Coded',
    'NSLC-BARR-0011': 'Complete/Hist Hard Coded',
    'NSLC-BARR-0017': 'Complete/Hist Hard Coded',
    'NSLC-BARR-0020': 'Complete/Diff Hard Coded',
    'NSLC-BARR-0021': 'Complete/Diff Hard Coded',
    'NSLC-BARR-0028': 'Complete/Diff Hard Coded',
    'NSLC-BARR-0031': 'Complete/Diff Hard Coded',
    'NSLC-BARR-0033': 'Complete/Diff Hard Coded',
    'NSLC-COLF-0007': 'Complete/Hist Hard Coded',
    'NSLC-COLF-0027': 'Complete/Hist Hard Coded',
    'NSLC-COLF-0030': 'Complete/Hist Hard Coded',
    'NSLC-COLF-0032': 'Complete/Diff&Hist Coded',
    'NSLC-COLF-0036': 'Complete/Diff Hard Coded',
    'NSLC-COLF-0040': 'Complete/Diff Hard Coded',
    'NSLC-COLF-0044': 'Complete/Hist Hard Coded',
    'NSLC-COLF-0061': 'Complete/Diff Hard Coded',
    'NSLC-COLF-0072': 'Complete/Hist Hard Coded',
    'NSLC-COLF-0074': 'Complete/Diff&Hist Coded',
    'NSLC-COLF-0113': 'Complete/Hist Hard Coded',
    'NSLC-COLF-0122': 'Complete/Hist Hard Coded',
    'NSLC-COLF-0126': 'Complete/Diff Hard Coded',
    'NSLC-DESA-0009': 'Complete/Diff Hard Coded',
    'NSLC-DESA-0010': 'Complete/Diff Hard Coded',
    'NSLC-DESA-0012': 'Complete/Diff Hard Coded',
    'NSLC-DESA-0015': 'Complete/Diff Hard Coded',
    'NSLC-FVDL-0002': 'Complete/Hist Hard Coded',
    'NSLC-HADA-0002': 'Complete/Diff&Hist Coded',
    'NSLC-IARC-0004': 'Complete/Hist Hard Coded',
    'NSLC-IARC-0075': 'Complete/Hist Hard Coded',
    'NSLC-IARC-0081': 'Complete/Diff&Hist Coded',
    'NSLC-IARC-0109': 'Complete/Hist Hard Coded',
    'NSLC-IARC-0205': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0074': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0081': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0085': 'Complete/Hist Hard Coded',
    'NSLC-LAVA-0121': 'Complete/Diff&Hist Coded',
    'NSLC-MDGM-0004': 'Complete/Diff Hard Coded',
    'NSLC-MDGM-0034': 'Complete/Hist Hard Coded',
    'NSLC-NICE-0054': 'Complete/Hist Hard Coded',
    'NSLC-NICE-0104': 'Complete/Diff&Hist Coded',
    'NSLC-PERU-0007': 'Complete/Diff Hard Coded',
    'NSLC-PERU-0008': 'Complete/Diff Hard Coded',
    'NSLC-PERU-0010': 'Complete/Diff Hard Coded',
    'NSLC-PERU-0014': 'Complete/Diff Hard Coded',
    'NSLC-PERU-0023': 'Complete/Hist Hard Coded',
    'NSLC-TWAN-0005': 'Complete/Diff Hard Coded',
    'NSLC-TWAN-0089': 'Complete/Hist Hard Coded',
    'NSLC-VLEN-0010': 'Complete/Hist Hard Coded'
}

# ✅ Replace Resolution column values
df['Resolution'] = df['name'].map(resolution_map).fillna(df['Resolution'])
print(df['Resolution'].value_counts(dropna=False))

In [ ]:
# Mapping from name -> Differentiation_result
# These ovverrides are optional and below are the ones that we had done in our study please adjust as needed
diff_result_map = {
    'NSLC-FVDL-0006': 'Not Calculated',
    'NSLC-IARC-0001': 'Not Calculated',
    'NSLC-IARC-0033': 'Not Calculated',
    'NSLC-IARC-0043': 'poorly',
    'NSLC-IARC-0045': 'Not Calculated',
    'NSLC-IARC-0073': 'poorly',
    'NSLC-IARC-0092': 'Not Calculated',
    'NSLC-IARC-0094': 'Not Calculated',
    'NSLC-IARC-0116': 'Poorly',
    'NSLC-IARC-0151': 'Not Calculated',
    'NSLC-IARC-0166': 'Not Calculated',
    'NSLC-ITLU-0021': 'Not Calculated',
    'NSLC-ITLU-0024': 'Not Calculated',
    'NSLC-LAVA-0009': 'Not Calculated',
    'NSLC-LAVA-0020': 'Not Calculated',
    'NSLC-LAVA-0027': 'Not Calculated',
    'NSLC-LAVA-0033': 'Not Calculated',
    'NSLC-LAVA-0035': 'Not Calculated',
    'NSLC-LAVA-0044': 'Not Calculated',
    'NSLC-LAVA-0056': 'Not Calculated',
    'NSLC-LAVA-0062': 'Not Calculated',
    'NSLC-LAVA-0092': 'Not Calculated',
    'NSLC-LAVA-0093': 'Not Calculated',
    'NSLC-LAVA-0100': 'well',
    'NSLC-LAVA-0115': 'moderate',
    'NSLC-MAYO-0075': 'Not Calculated',
    'NSLC-MOFF-0003': 'moderate',
    'NSLC-NICE-0001': 'Not Calculated',
    'NSLC-NICE-0022': 'Not Calculated',
    'NSLC-NICE-0045': 'Not Calculated',
    'NSLC-NICE-0056': 'Not Calculated',
    'NSLC-NICE-0101': 'Not Calculated',
    'NSLC-RONT-0056': 'Not Calculated',
    'NSLC-TWAN-0008': 'poorly',
    'NSLC-TWAN-0047': 'Not Calculated',
    'NSLC-TWAN-0064': 'Not Calculated',
    'NSLC-TWAN-0076': 'Not Calculated',
    'NSLC-TWAN-0086': 'Not Calculated',
    'NSLC-TWAN-0115': 'Not Calculated',
    'NSLC-TWAN-0148': 'Not Calculated',
    'NSLC-TWAN-0154': 'poorly',
    'NSLC-TWAN-0159': 'Not Calculated',
    'NSLC-TWAN-0205': 'Moderate',
    'NSLC-YALE-0003': 'poorly',
    'NSLC-YALE-0021': 'poorly',
    'NSLC-YALE-0032': 'poorly',
    'NSLC-BARR-0010': 'poorly',
    'NSLC-FVDL-0021': 'poorly',
    'NSLC-HARV-0017': 'moderate',
    'NSLC-IARC-0080': 'moderate',
    'NSLC-LAVA-0123': 'moderate',
    'NSLC-MAYO-0012': 'well',
    'NSLC-MAYO-0050': 'well',
    'NSLC-MAYO-0073': 'moderate',
    'NSLC-MAYO-0087': 'well',
    'NSLC-MDGM-0026': 'well',
    'NSLC-MDGM-0035': 'well',
    'NSLC-NICE-0023': 'well',
    'NSLC-NICE-0068': 'poorly',
    'NSLC-NICE-0080': 'well',
    'NSLC-NICE-0081': 'moderate',
    'NSLC-NICE-0087': 'moderate',
    'NSLC-NICE-0092': 'well',
    'NSLC-NICE-0102': 'well',
    'NSLC-NICE-0107': 'moderate',
    'NSLC-RONT-0002': 'moderate',
    'NSLC-RONT-0093': 'moderate',
    'NSLC-TWAN-0069': 'poorly',
    'NSLC-TWAN-0116': 'Not Calculated',
    'NSLC-VLEN-0019': 'poorly',
    'NSLC-VLEN-0020': 'moderate',
    'NSLC-YALE-0004': 'poorly',
    'NSLC-YALE-0010': 'well',
    'NSLC-YALE-0013': 'Moderate',
    'NSLC-YALE-0016': 'poorly',
    'NSLC-YALE-0023': 'moderate',
    'NSLC-YALE-0037': 'moderate',
    'NSLC-AUST-0004': 'Not Calculated',
    'NSLC-AUST-0006': 'moderate',
    'NSLC-AUST-0008': 'Not Calculated',
    'NSLC-AUST-0014': 'Not Calculated',
    'NSLC-AUST-0021': 'poorly',
    'NSLC-AUST-0023': 'Not Calculated',
    'NSLC-AUST-0036': 'Not Calculated',
    'NSLC-BARR-0009': 'Not Calculated',
    'NSLC-BARR-0011': 'Not Calculated',
    'NSLC-BARR-0017': 'Not Calculated',
    'NSLC-BARR-0020': 'Not Calculated',
    'NSLC-BARR-0021': 'Not Calculated',
    'NSLC-BARR-0028': 'Not Calculated',
    'NSLC-BARR-0031': 'Not Calculated',
    'NSLC-BARR-0033': 'Not Calculated',
    'NSLC-COLF-0007': 'Not Calculated',
    'NSLC-COLF-0027': 'Not Calculated',
    'NSLC-COLF-0030': 'Not Calculated',
    'NSLC-COLF-0032': 'Not Calculated',
    'NSLC-COLF-0036': 'Not Calculated',
    'NSLC-COLF-0040': 'Not Calculated',
    'NSLC-COLF-0044': 'Not Calculated',
    'NSLC-COLF-0061': 'Not Calculated',
    'NSLC-COLF-0072': 'Not Calculated',
    'NSLC-COLF-0074': 'Poorly',
    'NSLC-COLF-0113': 'Not Calculated',
    'NSLC-COLF-0122': 'poorly',
    'NSLC-COLF-0126': 'Not Calculated',
    'NSLC-DESA-0009': 'Not Calculated',
    'NSLC-DESA-0010': 'Poorly',
    'NSLC-DESA-0012': 'Not Calculated',
    'NSLC-DESA-0015': 'Not Calculated',
    'NSLC-FVDL-0002': 'Not Calculated',
    'NSLC-HADA-0002': 'moderate',
    'NSLC-IARC-0004': 'Not Calculated',
    'NSLC-IARC-0075': 'Not Calculated',
    'NSLC-IARC-0081': 'Not Calculated',
    'NSLC-IARC-0109': 'Not Calculated',
    'NSLC-IARC-0205': 'poorly',
    'NSLC-LAVA-0009': 'Not Calculated',
    'NSLC-LAVA-0074': 'Not Calculated',
    'NSLC-LAVA-0081': 'Not Calculated',
    'NSLC-LAVA-0085': 'Not Calculated',
    'NSLC-LAVA-0092': 'Not Calculated',
    'NSLC-LAVA-0121': 'well',
    'NSLC-MDGM-0004': 'moderate',
    'NSLC-MDGM-0034': 'Not Calculated',
    'NSLC-NICE-0054': 'Not Calculated',
    'NSLC-NICE-0101': 'Not Calculated',
    'NSLC-NICE-0104': 'Not Calculated',
    'NSLC-PERU-0007': 'Not Calculated',
    'NSLC-PERU-0008': 'Not Calculated',
    'NSLC-PERU-0010': 'poorly',
    'NSLC-PERU-0014': 'Not Calculated',
    'NSLC-PERU-0023': 'Not Calculated',
    'NSLC-TWAN-0005': 'Not Calculated',
    'NSLC-TWAN-0089': 'Not Calculated',
    'NSLC-VLEN-0010': 'Not Calculated',
}

# Apply the mapping; keep existing values when no match
df['Differentiation_result'] = df['name'].map(diff_result_map).fillna(df['Differentiation_result'])

standardize = {
    'Poorly': 'poorly',
    'Moderate': 'moderate',
    'Well': 'well',
    'Not calculated': 'Not Calculated'
}
df['Differentiation_result'] = df['Differentiation_result'].replace(standardize)

print(df['Differentiation_result'].value_counts(dropna=False))

In [ ]:
# 1️⃣ Keep only the 135 specified columns in the given order
# select columns as needed below are the one that we used
selected_columns = [
    'Status', 'Resolution', 'name', 'DCEG_Shrlck_study_dir_resolve',
    'P1', 'P2', 'P3', 'Fixassign',
    'Check1_P1', 'Check1_P2', 'Check1_P3',
    'header_comparison_test',
    'DCEG_Shlck_header_Diag_P1', 'DCEG_Shlck_header_Diag_P2', 'DCEG_Shlck_header_Diag_P3',
    'Primary_histology_comparison',
    'DCEG_Shrlck_HistoTypeSingle_P1', 'DCEG_Shrlck_HistoTypeSingle_P2', 'DCEG_Shrlck_HistoTypeSingle_P3',
    'DCEG_Shrlck_HistoTypeMixed1_P1', 'DCEG_Shrlck_HistoTypeMixed1_P2', 'DCEG_Shrlck_HistoTypeMixed1_P3',
    'DCEG_Shrlck_HistoTypeMixed2_P1', 'DCEG_Shrlck_HistoTypeMixed2_P2',
    'DCEG_Shrlck_HistoTypeNoEval_P1', 'DCEG_Shrlck_HistoTypeNoEval_P2', 'DCEG_Shrlck_HistoTypeNoEval_P3',
    'Differentiation_result', 'combine_differentiation', 'Compare_differentiation',
    'DifferentiationP1', 'DifferentiationP2', 'DifferentiationP3',
    'Adeno_filter',
    'getTotal_subtype_p1', 'getTotal_subtype_p2', 'getTotal_subtype_p3',
    'DCEG_Shrlck_add_dis_reason_P1', 'DCEG_Shrlck_add_dis_reason_P2', 'DCEG_Shrlck_add_dis_reason_P3',
    'NumberofPath', 'Pathologist',
    'Acinar+Pap_P1', 'Acinar+pap_P2',
    'High_gradeP1', 'High_gradeP2',
    'DCEG_Shlck_header_HistoSub_P1', 'DCEG_Shlck_header_HistoSub_P2',
    'DCEG_Shlck_microenv_P1', 'DCEG_Shlck_microenv_P2',
    'DCEG_Shrlck_HistoTypeMx1Com_P1', 'DCEG_Shrlck_HistoTypeMx1Com_P2',
    'DCEG_Shrlck_HistoTypeMx2Com_P1', 'DCEG_Shrlck_HistoTypeMx2Com_P2',
    'DCEG_Shrlck_HistoTypeMxCom_P1', 'DCEG_Shrlck_HistoTypeMxCom_P2',
    'DCEG_Shrlck_HistoTypeNECom_P1', 'DCEG_Shrlck_HistoTypeNECom_P2',
    'DCEG_Shrlck_HistoTypeSglCom_P1', 'DCEG_Shrlck_HistoTypeSglCom_P2',
    'DCEG_Shrlck_Lymphoidagg_P1', 'DCEG_Shrlck_Lymphoidagg_P2',
    'DCEG_Shrlck_STAS_P1', 'DCEG_Shrlck_STAS_P2',
    'DCEG_Shrlck_TILS_P1', 'DCEG_Shrlck_TILS_P2',
    'DCEG_Shrlck_add_discussion_P1', 'DCEG_Shrlck_add_discussion_P2',
    'DCEG_Shrlck_cyto_ClearCell_P1', 'DCEG_Shrlck_cyto_ClearCell_P2',
    'DCEG_Shrlck_cyto_Morular_P1', 'DCEG_Shrlck_cyto_Morular_P2',
    'DCEG_Shrlck_cyto_Mucinous_P1', 'DCEG_Shrlck_cyto_Mucinous_P2',
    'DCEG_Shrlck_cyto_SignetRing_P1', 'DCEG_Shrlck_cyto_SignetRing_P2',
    'DCEG_Shrlck_cyto_comment_P1', 'DCEG_Shrlck_cyto_comment_P2',
    'DCEG_Shrlck_cyto_fetal_P1', 'DCEG_Shrlck_cyto_fetal_P2',
    'DCEG_Shrlck_cytofeat_adeno_P1', 'DCEG_Shrlck_cytofeat_adeno_P2',
    'DCEG_Shrlck_finish_P1', 'DCEG_Shrlck_finish_P2',
    'DCEG_Shrlck_review_complete_P1', 'DCEG_Shrlck_review_complete_P2',
    'DCEG_Shrlck_subt_acinar_pct_P1', 'DCEG_Shrlck_subt_acinar_pct_P2',
    'DCEG_Shrlck_subt_comment_P1', 'DCEG_Shrlck_subt_comment_P2',
    'DCEG_Shrlck_subt_crib_pct_P1', 'DCEG_Shrlck_subt_crib_pct_P2',
    'DCEG_Shrlck_subt_lepdic_pct_P1', 'DCEG_Shrlck_subt_lepdic_pct_P2',
    'DCEG_Shrlck_subt_mpap_pct_P1', 'DCEG_Shrlck_subt_mpap_pct_P2',
    'DCEG_Shrlck_subt_pap_pct_P1', 'DCEG_Shrlck_subt_pap_pct_P2',
    'DCEG_Shrlck_subt_solid_pct_P1', 'DCEG_Shrlck_subt_solid_pct_P2',
    'Acinar+pap_P3', 'High_gradeP3',
    'DCEG_Shlck_header_HistoSub_P3', 'DCEG_Shlck_microenv_P3',
    'DCEG_Shrlck_HistoTypeMixed2_P3', 'DCEG_Shrlck_HistoTypeMx1Com_P3',
    'DCEG_Shrlck_HistoTypeMx2Com_P3', 'DCEG_Shrlck_HistoTypeSglCom_P3',
    'DCEG_Shrlck_Lymphoidagg_P3', 'DCEG_Shrlck_STAS_P3', 'DCEG_Shrlck_TILS_P3',
    'DCEG_Shrlck_add_discussion_P3', 'DCEG_Shrlck_cyto_ClearCell_P3',
    'DCEG_Shrlck_cyto_Morular_P3', 'DCEG_Shrlck_cyto_Mucinous_P3',
    'DCEG_Shrlck_cyto_SignetRing_P3', 'DCEG_Shrlck_cyto_comment_P3',
    'DCEG_Shrlck_cyto_fetal_P3', 'DCEG_Shrlck_cytofeat_adeno_P3',
    'DCEG_Shrlck_finish_P3', 'DCEG_Shrlck_review_complete_P3',
    'DCEG_Shrlck_subt_acinar_pct_P3', 'DCEG_Shrlck_subt_crib_pct_P3',
    'DCEG_Shrlck_subt_lepdic_pct_P3', 'DCEG_Shrlck_subt_mpap_pct_P3',
    'DCEG_Shrlck_subt_pap_pct_P3', 'DCEG_Shrlck_subt_solid_pct_P3',
    'DCEG_Shrlck_HistoTypeMxCom_P3', 'DCEG_Shrlck_HistoTypeNECom_P3',
    'DCEG_Shrlck_subt_comment_P3',
    'Diss_P1', 'Diss_P2', 'Diss_P3',
    'getFixative', 'Phase'
]

# Select only those columns in that order (ignore missing safely)
df_final = df[[c for c in selected_columns if c in df.columns]]


In [ ]:
#check the final columns
print(df_final.columns.tolist())


In [ ]:
# get the shape of the final dataframe
df_final.shape

In [ ]:
#write the final output to a csv file
df_final.to_csv('Pythonoutput.csv', index=False)